# Hybrid Ensemble — Fusing Bandit, Transformer Classifier, and RL

Implements the paper's Fusion Mechanism (Section IV-E): combines the three
independent components into one weighted ensemble score:

```
ensemble_score = (w_Bandit*P_Bandit + w_LLM*P_LLM + w_RL*P_RL) / (w_Bandit + w_LLM + w_RL)
```

Weights are **not fixed a priori** -- determined empirically via grid
search on the **validation** split, maximizing F1. The decision threshold
is likewise tuned on validation only (`argmax(F1)`), and the test split is
touched exactly once, at the very end.

**This notebook does not train anything new** -- it loads the per-sample
scores already saved by `Bandit.ipynb`, `RL.ipynb`, and `LLMs.ipynb`, and
combines them. Run those three first.

**Component selection**: `LLMs.ipynb` may have produced up to three
transformer models (CodeBERT, GraphCodeBERT, CodeT5) and `RL.ipynb`
produces two RL variants (DQN, Tabular Q-Learning). Consistent with the
paper's three-component formula (one Bandit + one LLM + one RL term),
this notebook automatically selects whichever transformer model and
whichever RL variant scored highest AUC on the **validation** set, and
reports which ones were chosen.

**Outputs saved:**
- `models/hybrid_scores.csv` -- per-sample ensemble score for every split
- `models/hybrid_config.json` -- chosen components, weights, threshold (used by `app.py` for live predictions)
- `results/hybrid_metrics.json` -- final test-set metrics
- `results/hybrid_roc_curve.png`, `results/hybrid_confusion_matrix.png`


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, classification_report)

warnings.filterwarnings('ignore')
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

CSV = 'dataset_with_content.csv'
if not os.path.exists(CSV):
    raise FileNotFoundError('Run the dataset-collection notebook first.')
df = pd.read_csv(CSV)
df.columns = df.columns.str.strip().str.lower()
if 'id' not in df.columns:
    df['id'] = df.index
print(f'Dataset: {len(df):,} samples')


## Load every available component's per-sample scores

Each of `Bandit.ipynb` / `RL.ipynb` / `LLMs.ipynb` saves a small CSV with
`id, split, <component>_score_column`. Missing files are skipped with a
clear warning rather than crashing -- you can build a partial ensemble
(e.g. Bandit + one transformer only) if not everything has been run yet.


In [ ]:
# ── Load component score files ────────────────────────────────────────────
COMPONENT_FILES = {
    'bandit_score':        ('models/bandit_scores.csv',        'bandit_score'),
    'codebert_prob':       ('models/codebert_scores.csv',      'codebert_prob'),
    'graphcodebert_prob':  ('models/graphcodebert_scores.csv', 'graphcodebert_prob'),
    'codegpt_prob':        ('models/codegpt_scores.csv',       'codegpt_prob'),
    'dqn_prob':            ('models/rl_scores.csv',            'dqn_prob'),
    'qlearn_prob':         ('models/rl_scores.csv',            'qlearn_prob'),
}

loaded = {}
for col_name, (path, col) in COMPONENT_FILES.items():
    if not os.path.exists(path):
        print(f'  [missing] {col_name:20s} <- {path} (not found, skipping)')
        continue
    sdf = pd.read_csv(path)
    if col not in sdf.columns:
        print(f'  [missing] {col_name:20s} <- {path} exists but has no column "{col}"')
        continue
    loaded[col_name] = sdf[['id', 'split', col]].rename(columns={col: col_name})
    print(f'  [ok]      {col_name:20s} <- {path}  ({len(sdf):,} rows)')

if 'bandit_score' not in loaded:
    raise RuntimeError('Bandit scores are required for the ensemble (paper requires all '
                        'three components). Run Bandit.ipynb first.')
llm_candidates = [c for c in ['codebert_prob', 'graphcodebert_prob', 'codegpt_prob'] if c in loaded]
rl_candidates = [c for c in ['dqn_prob', 'qlearn_prob'] if c in loaded]
if not llm_candidates:
    raise RuntimeError('No transformer scores found. Run Transformer.ipynb first.')
if not rl_candidates:
    raise RuntimeError('No RL scores found. Run RL.ipynb first.')

print(f'\nAvailable LLM candidates: {llm_candidates}')
print(f'Available RL candidates: {rl_candidates}')


In [ ]:
# ── Merge all components on (id, split) ──────────────────────────────────
merged = df[['id', 'label']].copy()
for col_name, sdf in loaded.items():
    merged = merged.merge(sdf, on='id', how='inner')
    if 'split' not in merged.columns:
        merged = merged.rename(columns={'split_x': 'split'}).drop(columns=['split_y'], errors='ignore')
    else:
        # keep a single 'split' column (all sources should agree on the same splits)
        split_cols = [c for c in merged.columns if c.startswith('split')]
        if len(split_cols) > 1:
            merged['split'] = merged[split_cols[0]]
            merged = merged.drop(columns=[c for c in split_cols if c != 'split'])

val_df = merged[merged['split'] == 'val'].copy()
test_df = merged[merged['split'] == 'test'].copy()
print(f'Merged rows with ALL required components -- val: {len(val_df):,}, test: {len(test_df):,}')

y_val, y_test = val_df['label'].values, test_df['label'].values


## Select the best LLM and RL component (by validation AUC)


In [ ]:
# ── Auto-select representative LLM and RL components ─────────────────────
def val_auc(col):
    return roc_auc_score(y_val, val_df[col].values)

llm_aucs = {c: val_auc(c) for c in llm_candidates}
rl_aucs = {c: val_auc(c) for c in rl_candidates}
print('LLM candidate AUCs (validation):', {k: round(v, 4) for k, v in llm_aucs.items()})
print('RL candidate AUCs (validation):', {k: round(v, 4) for k, v in rl_aucs.items()})

best_llm = max(llm_aucs, key=llm_aucs.get)
best_rl = max(rl_aucs, key=rl_aucs.get)
print(f'\nSelected LLM component for the ensemble: {best_llm} (AUC={llm_aucs[best_llm]:.4f})')
print(f'Selected RL component for the ensemble: {best_rl} (AUC={rl_aucs[best_rl]:.4f})')


## Grid search fusion weights + decision threshold (validation only)


In [ ]:
# ── Joint search over weights AND threshold together, using the SAME ─────
# asymmetric reward table (Table II) already used to train RL -- not F1. ──
# This keeps one consistent objective across the whole paper (RL training,
# weight selection, threshold selection) instead of switching to a
# different external metric (F1) only at the ensemble-fusion step.
REWARD = {(1, 1): 2.0, (0, 0): 1.5, (0, 1): -1.0, (1, 0): -1.3}  # Table II, unchanged

def mean_reward(y_true, preds):
    return np.mean([REWARD[(int(yt), int(p))] for yt, p in zip(y_true, preds)])

bandit_val, llm_val, rl_val = val_df['bandit_score'].values, val_df[best_llm].values, val_df[best_rl].values

best_reward, best_weights, best_thr = -999.0, (0.34, 0.33, 0.33), 0.5
weight_grid = np.arange(0.0, 1.01, 0.05)       # 0.0 allowed -- true single-component extremes included
thr_grid_coarse = np.arange(0.05, 0.96, 0.05)  # coarse inner loop for speed

for wb in weight_grid:
    for wl in weight_grid:
        wr = round(1.0 - wb - wl, 3)
        if wr < -1e-9 or wr > 1.0 + 1e-9:
            continue
        total = wb + wl + wr
        if total <= 0:
            continue
        ens = (wb * bandit_val + wl * llm_val + wr * rl_val) / total
        for thr in thr_grid_coarse:
            preds = (ens >= thr).astype(int)
            r = mean_reward(y_val, preds)
            if r > best_reward:
                best_reward, best_weights, best_thr = r, (round(wb, 3), round(wl, 3), round(wr, 3)), thr

w_bandit, w_llm, w_rl = best_weights
print(f'Best joint (weights, threshold) on coarse grid -- '
      f'Bandit: {w_bandit}, LLM ({best_llm}): {w_llm}, RL ({best_rl}): {w_rl}, '
      f'threshold: {best_thr:.2f}')
print(f'Validation mean reward (Table II) at this combination: {best_reward:.4f}')

# ── Refine the threshold at finer resolution around the best point found ──
total_w = w_bandit + w_llm + w_rl if (w_bandit + w_llm + w_rl) > 0 else 1.0
ens_val = (w_bandit * bandit_val + w_llm * llm_val + w_rl * rl_val) / total_w

best_thr_fine, best_reward_fine = best_thr, best_reward
for thr in np.arange(0.02, 0.99, 0.01):
    preds = (ens_val >= thr).astype(int)
    r = mean_reward(y_val, preds)
    if r > best_reward_fine:
        best_reward_fine, best_thr_fine = r, thr
best_thr = best_thr_fine
final_val_preds = (ens_val >= best_thr).astype(int)
best_thr_f1 = f1_score(y_val, final_val_preds, zero_division=0)  # reported for reference only

print(f'\nRefined decision threshold: {best_thr:.2f} '
      f'(validation mean reward={best_reward_fine:.4f}, F1={best_thr_f1:.4f})')

# ── Reference check: does the ensemble beat the strongest single component
#    alone, using the SAME reward criterion? Reported honestly either way. ──
best_single_reward, best_single_thr = -999.0, 0.5
for thr in np.arange(0.02, 0.99, 0.01):
    preds = (llm_val >= thr).astype(int)
    r = mean_reward(y_val, preds)
    if r > best_single_reward:
        best_single_reward, best_single_thr = r, thr
print(f'Reference -- {best_llm} ALONE at its own best threshold: '
      f'mean reward={best_single_reward:.4f} (threshold={best_single_thr:.2f})')
if best_reward_fine >= best_single_reward:
    print(f'Ensemble matches or beats the strongest single component on the Table II reward criterion.')
else:
    print(f'NOTE: the strongest single component alone still scores higher on the reward criterion '
          f'({best_single_reward:.4f} vs ensemble {best_reward_fine:.4f}) -- report this honestly.')


## Final test-set evaluation


In [ ]:
# ── Evaluate on the held-out test set (weights + threshold both from validation only) ──
bandit_test, llm_test, rl_test = test_df['bandit_score'].values, test_df[best_llm].values, test_df[best_rl].values
ens_test = (w_bandit * bandit_test + w_llm * llm_test + w_rl * rl_test) / total_w
test_preds = (ens_test >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
prec = precision_score(y_test, test_preds, zero_division=0)
rec = recall_score(y_test, test_preds, zero_division=0)
f1 = f1_score(y_test, test_preds, zero_division=0)
auc_val = roc_auc_score(y_test, ens_test)
cm = confusion_matrix(y_test, test_preds)
tn, fp, fn, tp = cm.ravel()

print('=== Hybrid Ensemble — Test Set ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Score  : {f1:.4f}')
print(f'AUC-ROC   : {auc_val:.4f}')
print()
print(classification_report(y_test, test_preds, target_names=['Safe', 'Vulnerable']))


In [ ]:
# ── ROC curve + confusion matrix ─────────────────────────────────────────
BG, AXB, CT, CG = '#0f172a', '#1e293b', '#f1f5f9', '#334155'

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor=BG)

ax = axes[0]
ax.set_facecolor(AXB)
fpr, tpr, _ = roc_curve(y_test, ens_test)
ax.plot(fpr, tpr, color='#a855f7', linewidth=2.5, label=f'Hybrid Ensemble (AUC = {auc_val:.4f})')
ax.plot([0, 1], [0, 1], linestyle='--', color=CG, label='Random guess (AUC = 0.50)')
ax.set_xlabel('False Positive Rate', color=CT); ax.set_ylabel('True Positive Rate', color=CT)
ax.set_title('Hybrid Ensemble — ROC Curve', color=CT, fontweight='bold')
ax.tick_params(colors=CT); ax.legend(loc='lower right', facecolor=AXB, labelcolor=CT)
ax.grid(alpha=0.2, color=CG)

ax2 = axes[1]
ax2.set_facecolor(AXB)
sns.heatmap(cm, annot=True, fmt='d', ax=ax2, cmap='RdYlGn',
            xticklabels=['Safe', 'Vulnerable'], yticklabels=['Safe', 'Vulnerable'],
            cbar=False, annot_kws={'size': 14, 'color': 'white'})
ax2.set_title('Confusion Matrix', color=CT, fontweight='bold')
ax2.set_xlabel('Predicted', color='#94a3b8'); ax2.set_ylabel('Actual', color='#94a3b8')
ax2.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('results/hybrid_roc_curve.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.savefig('results/hybrid_confusion_matrix.png', dpi=300, facecolor=BG, bbox_inches='tight')
plt.show()
print('Saved: results/hybrid_roc_curve.png, results/hybrid_confusion_matrix.png')


## Save everything (scores, config for live inference, metrics)


In [ ]:
# ── Save per-sample ensemble scores for every split ──────────────────────
all_ens = []
for split_name, sdf in [('train', merged[merged['split'] == 'train']),
                         ('val', val_df), ('test', test_df)]:
    if len(sdf) == 0:
        continue
    ens = (w_bandit * sdf['bandit_score'].values + w_llm * sdf[best_llm].values +
           w_rl * sdf[best_rl].values) / total_w
    all_ens.append(pd.DataFrame({'id': sdf['id'].values, 'split': split_name, 'ensemble_score': ens}))
hybrid_scores_df = pd.concat(all_ens, ignore_index=True)
hybrid_scores_df.to_csv('models/hybrid_scores.csv', index=False)
print(f'Saved: models/hybrid_scores.csv ({len(hybrid_scores_df):,} rows)')

# ── Save the ensemble config -- this is what app.py needs for LIVE predictions ──
hybrid_config = {
    'llm_component': best_llm,
    'llm_model_key': best_llm.replace('_prob', ''),
    'rl_component': best_rl,
    'weights': {'bandit': w_bandit, 'llm': w_llm, 'rl': w_rl},
    'threshold': round(float(best_thr), 4),
}
with open('models/hybrid_config.json', 'w') as f:
    json.dump(hybrid_config, f, indent=2)
print(f'Saved: models/hybrid_config.json -> {hybrid_config}')

# ── Save metrics ──────────────────────────────────────────────────────────
metrics = {
    'method': f'Hybrid Ensemble (Bandit + {best_llm} + {best_rl})',
    'weights': hybrid_config['weights'],
    'threshold': hybrid_config['threshold'],
    'accuracy': round(float(acc), 4), 'precision': round(float(prec), 4),
    'recall': round(float(rec), 4), 'f1': round(float(f1), 4), 'auc': round(float(auc_val), 4),
    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
    'test_size': int(len(y_test)),
}
with open('results/hybrid_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/hybrid_metrics.json')
print(json.dumps(metrics, indent=2))


## Next steps

1. `app.py`'s ROC tab will now pick up `models/hybrid_scores.csv` and
   `results/hybrid_metrics.json` automatically.
2. Use `inference_utils.py` (separate file) for **live** predictions on
   user-pasted code in `app.py` -- it reads `models/hybrid_config.json`
   saved above to know which exact LLM/RL components and weights to use.
3. Update Table IV / Section VI with these real, honestly-derived ensemble
   numbers.


## H3 Evidence — Time-to-Detection (TTD) and Adaptive Pipeline Overhead

Directly addresses the reviewer's second, separate request: *"Measure
TTD/TTR and pipeline overhead so H3 has evidence."*

**What H3 actually claims**: applying RL for adaptive scanning decisions
reduces CI/CD pipeline execution time while maintaining detection
performance. The mechanism that could make this true is **adaptive
gating**: Bandit + RL are both fast (rule matching / an 11-feature forward
pass); the transformer model is comparatively slow (a full forward pass
through a 100M+ parameter network). If Bandit+RL already agree with high
confidence, there is no need to also pay for the expensive transformer
call.

This section **measures** that directly -- real wall-clock timings on
this machine, not simulated numbers -- comparing two pipelines on the
same test samples:
- **Baseline**: always runs Bandit + RL + the transformer model.
- **Adaptive**: only calls the transformer model when the fast
  Bandit+RL path is uncertain (within `CONFIDENCE_MARGIN` of the decision
  boundary).

**Honest scope note on TTR**: Time-to-Remediation (how long a *human*
takes to actually fix a flagged vulnerability after deployment) cannot be
measured from this offline study -- it requires live deployment data
tracking real fix turnaround in an organization, which this project does
not have. What follows is Time-to-Detection (system latency) and pipeline
overhead specifically, which is what's actually measurable here and, per
the reviewer's own note, what a DevSecOps audience cares about most.


In [ ]:
# ── H3: Time-to-Detection + adaptive pipeline overhead ───────────────────
import time
import sys
sys.path.insert(0, '.')
import inference_utils as iu

N_TIMING_SAMPLES = min(100, len(test_df))
CONFIDENCE_MARGIN = 0.15   # how close to the threshold counts as "uncertain"

timing_ids = test_df['id'].sample(N_TIMING_SAMPLES, random_state=42).tolist()
content_by_id = df.set_index('id')['content']
label_by_id = df.set_index('id')['label']

llm_display_map = {'codebert_prob': 'CodeBERT', 'graphcodebert_prob': 'GraphCodeBERT',
                    'codegpt_prob': 'CodeGPT'}
rl_display_map = {'dqn_prob': 'DQN', 'qlearn_prob': 'Q-Learning'}
chosen_llm_display = llm_display_map[best_llm]
chosen_rl_display = rl_display_map[best_rl]

per_component_times = {'bandit': [], 'rl': [], 'llm': []}
baseline_times, adaptive_times, used_llm_flags = [], [], []
baseline_preds, adaptive_preds, y_timing = [], [], []

print(f'Timing {N_TIMING_SAMPLES} real samples through both pipelines '
      f'(baseline = always full scan, adaptive = RL+Bandit-gated)...')
for cid in timing_ids:
    code_str = content_by_id.loc[cid]
    y_timing.append(int(label_by_id.loc[cid]))

    t0 = time.perf_counter()
    b = iu.bandit_score_live(code_str)
    t_bandit = time.perf_counter() - t0

    t1 = time.perf_counter()
    rl_scores = iu.rl_predict_live(code_str)
    rl = rl_scores.get(chosen_rl_display)
    rl = rl if rl is not None else 0.5
    t_rl = time.perf_counter() - t1

    t_fast = t_bandit + t_rl
    fast_score = (b + rl) / 2   # simple average, used ONLY for the gating decision

    t2 = time.perf_counter()
    llm_scores = iu.transformer_predict_live(code_str)
    llm = llm_scores.get(chosen_llm_display)
    llm = llm if llm is not None else 0.5
    t_llm = time.perf_counter() - t2

    per_component_times['bandit'].append(t_bandit)
    per_component_times['rl'].append(t_rl)
    per_component_times['llm'].append(t_llm)

    full_score = (w_bandit * b + w_llm * llm + w_rl * rl) / total_w
    baseline_times.append(t_fast + t_llm)
    baseline_preds.append(int(full_score >= best_thr))

    if abs(fast_score - 0.5) < CONFIDENCE_MARGIN:
        adaptive_times.append(t_fast + t_llm)
        used_llm_flags.append(True)
        adaptive_score = full_score
    else:
        adaptive_times.append(t_fast)
        used_llm_flags.append(False)
        adaptive_score = fast_score
    adaptive_preds.append(int(adaptive_score >= best_thr))

y_timing = np.array(y_timing)
baseline_total, adaptive_total = sum(baseline_times), sum(adaptive_times)
pct_time_saved = (1 - adaptive_total / baseline_total) * 100 if baseline_total > 0 else 0.0
pct_skipped_llm = (1 - sum(used_llm_flags) / len(used_llm_flags)) * 100

baseline_acc = accuracy_score(y_timing, baseline_preds)
adaptive_acc = accuracy_score(y_timing, adaptive_preds)
baseline_f1 = f1_score(y_timing, baseline_preds, zero_division=0)
adaptive_f1 = f1_score(y_timing, adaptive_preds, zero_division=0)

print('\n=== Per-component Time-to-Detection (TTD), ms/sample ===')
for comp, times_list in per_component_times.items():
    arr = np.array(times_list) * 1000
    print(f'  {comp:8s}: mean={arr.mean():.2f}ms  median={np.median(arr):.2f}ms  '
          f'std={arr.std():.2f}ms')

print(f'\n=== Pipeline totals over {N_TIMING_SAMPLES} samples ===')
print(f'Baseline (always full scan) : {baseline_total*1000:.1f} ms total '
      f'({baseline_total/N_TIMING_SAMPLES*1000:.2f} ms/sample avg)')
print(f'Adaptive (RL+Bandit-gated)  : {adaptive_total*1000:.1f} ms total '
      f'({adaptive_total/N_TIMING_SAMPLES*1000:.2f} ms/sample avg)')
print(f'Time saved by adaptive gating: {pct_time_saved:.1f}%')
print(f'Transformer calls skipped   : {pct_skipped_llm:.1f}% of samples')

print(f'\n=== Detection performance: baseline vs adaptive ===')
print(f'Baseline : Accuracy={baseline_acc:.4f}  F1={baseline_f1:.4f}')
print(f'Adaptive : Accuracy={adaptive_acc:.4f}  F1={adaptive_f1:.4f}')
print(f'Performance delta: Accuracy {adaptive_acc-baseline_acc:+.4f}, F1 {adaptive_f1-baseline_f1:+.4f}')

h3_result = {
    'n_samples': int(N_TIMING_SAMPLES),
    'confidence_margin': CONFIDENCE_MARGIN,
    'ttd_ms_per_sample': {
        comp: {'mean': round(float(np.mean(t)*1000), 3), 'median': round(float(np.median(t)*1000), 3),
               'std': round(float(np.std(t)*1000), 3)}
        for comp, t in per_component_times.items()
    },
    'pipeline_overhead': {
        'baseline_total_ms': round(float(baseline_total*1000), 2),
        'adaptive_total_ms': round(float(adaptive_total*1000), 2),
        'pct_time_saved': round(float(pct_time_saved), 2),
        'pct_transformer_calls_skipped': round(float(pct_skipped_llm), 2),
    },
    'detection_performance': {
        'baseline': {'accuracy': round(float(baseline_acc), 4), 'f1': round(float(baseline_f1), 4)},
        'adaptive': {'accuracy': round(float(adaptive_acc), 4), 'f1': round(float(adaptive_f1), 4)},
    },
    'ttr_note': ('Time-to-Remediation requires live deployment data tracking real human '
                 'fix turnaround, which this offline study does not have -- not reported.'),
}
with open('results/h3_ttd_pipeline_overhead.json', 'w') as f:
    json.dump(h3_result, f, indent=2)
print('\nSaved: results/h3_ttd_pipeline_overhead.json')
